# Module 04 — Lifecycle Hooks

Hooks let you run custom code at specific points in the agent's lifecycle — without modifying the agent's core logic.

### Available hook events

| Event | When it fires |
|---|---|
| `AgentInitializedEvent` | Once, right after the agent is created |
| `MessageAddedEvent` | Every time a message is added to the conversation |
| `AfterInvocationEvent` | After each call to `agent(...)` completes |

### Why hooks?

Hooks keep concerns separated:
- **Logging** — track every message without cluttering agent code
- **Memory** — load preferences on init, save conversations after each turn
- **Guardrails** — inspect messages before they're processed
- **Analytics** — count tokens, measure latency, track tool usage

In [2]:
from strands import Agent
from strands.hooks import (
    AgentInitializedEvent,
    MessageAddedEvent,
    AfterInvocationEvent,
    HookProvider,
    HookRegistry
)

### Simple logging hook

The simplest hook just logs what's happening.

In [3]:
class LoggingHook(HookProvider):
    """Logs agent lifecycle events to the console."""
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        print(f"[HOOK] Agent '{event.agent.name}' initialized")
    
    def on_message_added(self, event: MessageAddedEvent):
        role = event.message.get('role', 'unknown')
        content = event.message.get('content', [])
        text = content[0].get('text', '')[:60] if content else ''
        print(f"[HOOK] Message added — role={role}: {text!r}")
    
    def on_after_invocation(self, event: AfterInvocationEvent):
        msg_count = len(event.agent.messages)
        print(f"[HOOK] Invocation complete — {msg_count} messages in history")
    
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)


agent = Agent(
    name="LoggedAgent",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    hooks=[LoggingHook()]
)

agent("What is the capital of India?")

[HOOK] Agent 'LoggedAgent' initialized
[HOOK] Message added — role=user: 'What is the capital of India?'
The capital of India is New Delhi. It is located in the northern part of the country and has been the capital since 1931, replacing Kolkata.[HOOK] Message added — role=assistant: 'The capital of India is New Delhi. It is located in the nort'
[HOOK] Invocation complete — 2 messages in history


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': 'The capital of India is New Delhi. It is located in the northern part of the country and has been the capital since 1931, replacing Kolkata.'}], 'metadata': {'usage': {'inputTokens': 14, 'outputTokens': 36, 'totalTokens': 50}, 'metrics': {'latencyMs': 1075, 'timeToFirstByteMs': 1029}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[2.127326011657715], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='b1a5a246-3d7b-4c26-b850-71e067b1fa75', usage={'inputTokens': 14, 'outputTokens': 36, 'totalTokens': 50})], usage={'inputTokens': 14, 'outputTokens': 36, 'totalTokens': 50})], traces=[<strands.telemetry.metrics.Trace object at 0x110a8e270>], accumulated_usage={'inputTokens': 14, 'outputTokens': 36, 'totalTokens': 50}, accumulated_metrics={'latencyMs': 1075}), state={}, interrupts=None, structured_output=None)

### Context injection hook

A common pattern: use `AgentInitializedEvent` to inject context into the system prompt before the first message.

In [5]:
class ContextInjectionHook(HookProvider):
    """Injects user context into the system prompt on agent init."""
    
    def __init__(self, user_context: str):
        self.user_context = user_context
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        # Append context to the existing system prompt
        event.agent.system_prompt += f"\n\n## User Context:\n{self.user_context}"
        print(f"[HOOK] Injected context into system prompt")
    
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)


# Simulate context that might come from a database or memory store
user_context = """- Name: Yeshwanth
- Dietary restrictions: vegetarian
- Favorite cuisine: Japanese
- Dislikes: cilantro"""

agent = Agent(
    name="PersonalChef",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="You are a personal food assistant. Be concise.",
    hooks=[ContextInjectionHook(user_context)]
)

agent("Recommend a dish for me tonight.")

[HOOK] Injected context into system prompt
# Vegetable Tempura

A perfect choice for you! Light, crispy battered vegetables served with dipping sauce. It's a Japanese classic that's naturally vegetarian and cilantro-free.

**Why it works:** Easy to prepare, delicious, and hits your love for Japanese cuisine. Pair it with miso soup and rice for a complete meal.

Would you like the recipe or cooking tips?

AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "# Vegetable Tempura\n\nA perfect choice for you! Light, crispy battered vegetables served with dipping sauce. It's a Japanese classic that's naturally vegetarian and cilantro-free.\n\n**Why it works:** Easy to prepare, delicious, and hits your love for Japanese cuisine. Pair it with miso soup and rice for a complete meal.\n\nWould you like the recipe or cooking tips?"}], 'metadata': {'usage': {'inputTokens': 66, 'outputTokens': 89, 'totalTokens': 155}, 'metrics': {'latencyMs': 2032, 'timeToFirstByteMs': 5057}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[5.698941230773926], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='e1221d16-fa4f-4e2b-a113-b115fc7a0af1', usage={'inputTokens': 66, 'outputTokens': 89, 'totalTokens': 155})], usage={'inputTokens': 66, 'outputTokens': 89, 'totalTokens': 155})], traces=[<strands.telemetry.metrics.Tra

### Conversation saver hook

Use `AfterInvocationEvent` to persist conversations — this is the foundation of memory.

In [6]:
class ConversationSaverHook(HookProvider):
    """Saves conversation turns to a local list (simulating a database)."""
    
    def __init__(self):
        self.saved_turns = []
    
    def on_after_invocation(self, event: AfterInvocationEvent):
        messages = event.agent.messages
        if len(messages) < 2:
            return
        
        # Get the last user message and assistant response
        user_msg = None
        assistant_msg = None
        
        for msg in reversed(messages):
            if msg['role'] == 'assistant' and not assistant_msg:
                content = msg.get('content', [])
                if content and isinstance(content[0], dict):
                    assistant_msg = content[0].get('text', '')
            elif msg['role'] == 'user' and not user_msg:
                content = msg.get('content', [])
                if content and isinstance(content[0], dict):
                    user_msg = content[0].get('text', '')
                    break
        
        if user_msg and assistant_msg:
            self.saved_turns.append({"user": user_msg, "assistant": assistant_msg})
            print(f"[HOOK] Saved turn #{len(self.saved_turns)} to memory")
    
    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AfterInvocationEvent, self.on_after_invocation)


saver = ConversationSaverHook()

agent = Agent(
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="You are a helpful assistant. Be brief.",
    hooks=[saver]
)

agent("I love Thai food.")
agent("I'm allergic to Non-veg.")

print("\nSaved conversation turns:")
for i, turn in enumerate(saver.saved_turns, 1):
    print(f"\nTurn {i}:")
    print(f"  User     : {turn['user']}")
    print(f"  Assistant: {turn['assistant'][:80]}...")

Thai food is delicious! The combination of spicy, sweet, sour, and salty flavors is really appealing. Do you have a favorite dish—like pad thai, green curry, or tom yum? Or are you looking for recommendations?[HOOK] Saved turn #1 to memory
Got it! Thailand has great vegetarian options. Some popular non-meat dishes include:

- **Pad Thai** (with tofu instead of shrimp)
- **Green/Red/Yellow Curry** (vegetable-based)
- **Tom Yum** (vegetable soup)
- **Pad Krapow** (spicy basil stir-fry with tofu)
- **Spring Rolls** (vegetable)
- **Satay** (peanut sauce with vegetables)
- **Mango Sticky Rice** (dessert)

Most Thai restaurants are happy to make dishes vegetarian. Just ask them to substitute meat with tofu or extra vegetables![HOOK] Saved turn #2 to memory

Saved conversation turns:

Turn 1:
  User     : I love Thai food.
  Assistant: Thai food is delicious! The combination of spicy, sweet, sour, and salty flavors...

Turn 2:
  User     : I'm allergic to Non-veg.
  Assistant: Got it! Thailan

### Combining multiple hooks

Pass a list of hook providers — they all fire in order.

In [7]:
agent = Agent(
    name="MultiHookAgent",
    model="us.anthropic.claude-haiku-4-5-20251001-v1:0",
    system_prompt="You are a helpful assistant.",
    hooks=[
        LoggingHook(),
        ContextInjectionHook("User prefers short answers."),
        ConversationSaverHook()
    ]
)

agent("What's a good snack?")

[HOOK] Agent 'MultiHookAgent' initialized
[HOOK] Injected context into system prompt
[HOOK] Message added — role=user: "What's a good snack?"
Here are some quick options:

- **Nuts or seeds** – protein-packed and satisfying
- **Fruit** – apple, banana, or berries for natural sweetness
- **Yogurt** – creamy and filling
- **Cheese & crackers** – good balance of flavors
- **Popcorn** – light and crunchy
- **Hummus & veggies** – healthy and tasty

It depends on what you're craving—sweet, salty, or savory![HOOK] Message added — role=assistant: 'Here are some quick options:\n\n- **Nuts or seeds** – protein-'
[HOOK] Saved turn #1 to memory
[HOOK] Invocation complete — 2 messages in history


AgentResult(stop_reason='end_turn', message={'role': 'assistant', 'content': [{'text': "Here are some quick options:\n\n- **Nuts or seeds** – protein-packed and satisfying\n- **Fruit** – apple, banana, or berries for natural sweetness\n- **Yogurt** – creamy and filling\n- **Cheese & crackers** – good balance of flavors\n- **Popcorn** – light and crunchy\n- **Hummus & veggies** – healthy and tasty\n\nIt depends on what you're craving—sweet, salty, or savory!"}], 'metadata': {'usage': {'inputTokens': 32, 'outputTokens': 115, 'totalTokens': 147}, 'metrics': {'latencyMs': 1793, 'timeToFirstByteMs': 3323}}}, metrics=EventLoopMetrics(cycle_count=1, tool_metrics={}, cycle_durations=[4.512874126434326], agent_invocations=[AgentInvocation(cycles=[EventLoopCycleMetric(event_loop_cycle_id='79f7dc4d-0409-4b8c-8577-cf865d205806', usage={'inputTokens': 32, 'outputTokens': 115, 'totalTokens': 147})], usage={'inputTokens': 32, 'outputTokens': 115, 'totalTokens': 147})], traces=[<strands.telemetry.metr

---

### Key takeaways

- Implement `HookProvider` and override the event methods you need
- `register_hooks` wires your callbacks to the registry
- `AgentInitializedEvent` → inject context, load memory
- `AfterInvocationEvent` → save conversations, log metrics
- `MessageAddedEvent` → inspect/filter messages in real time
- Pass multiple hooks as a list — they compose cleanly

Next up: **05_memory.ipynb** — persistent memory with AWS AgentCore.